In [15]:
# Imports
import os
import pandas as pd
import gradio as gr
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from llama_index.core import PromptTemplate
from llama_index.llms.groq import Groq
from llama_index.experimental.query_engine.pandas import PandasInstructionParser
from llama_index.core.workflow import Workflow, step, Event, StartEvent, StopEvent



In [16]:
# Carrega as variáveis de ambiente
load_dotenv()
# Evita que a LlamaIndex trave procurando credenciais da OpenAI 
os.environ["OPENAI_API_KEY"] = "NA"


### Definição dos eventos e fluxo de trabalho

In [17]:
# 1. Definimos os Eventos
class PandasCodeEvent(Event):
    query_str: str
    pandas_instructions: str

class PandasOutputEvent(Event):
    query_str: str
    pandas_instructions: str
    pandas_output: str
    figura: object = None

# 2. Criamos o Fluxo de Trabalho
class PandasAnalistaWorkflow(Workflow):
    def __init__(self, df: pd.DataFrame, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.df = df
        
        self.llm = Groq(model="llama-3.3-70b-versatile", api_key=os.environ.get("GROQ_API_KEY"))

        self.instruction_str = (
            "1. Converta a consulta para código Python executável usando Pandas.\n"
            "2. A linha final do código deve ser uma expressão Python que possa ser chamada com a função `eval()`.\n"
            "3. O código deve representar uma solução para a consulta.\n"
            "4. IMPRIMA APENAS A EXPRESSÃO.\n"
            "5. Não coloque a expressão entre aspas.\n"
        )

        self.pandas_prompt = PromptTemplate(
            "Você está trabalhando com um dataframe do pandas em Python chamado `df`.\n"
            "{colunas_detalhes}\n\n"
            "Este é o resultado de `print(df.head())`:\n"
            "{df_str}\n\n"
            "Siga estas instruções:\n"
            "{instruction_str}\n"
            "Consulta: {query_str}\n\n"
            "Expressão:"
        )

        self.synthesis_prompt = PromptTemplate(
            "Dada uma pergunta de entrada, atue como analista de dados e elabore uma resposta a partir dos resultados da consulta.\n"
            "Responda de forma natural, sem introduções como 'A resposta é:' ou algo semelhante.\n"
            "Consulta: {query_str}\n\n"
            "Instruções do Pandas:\n{pandas_instructions}\n\n"
            "Saída do Pandas: {pandas_output}\n\n"
            "Resposta:\n\n"
            "Ao final, exibir o código usado em para gerar a resposta, no formato: O código utilizado foi `{pandas_instructions}`"
        )

    def descreve_colunas(self):
        descricao = "\n".join([f"`{col}`: {str(self.df[col].dtype)}" for col in self.df.columns])
        return "Detalhes das colunas do Dataframe:\n" + descricao

    # PASSO 1: Recebe a pergunta do usuário (StartEvent) e devolve o código (PandasCodeEvent)
    @step
    async def gerar_codigo_pandas(self, ev: StartEvent) -> PandasCodeEvent:
        query_str = ev.get("query")
        prompt_formatado = self.pandas_prompt.format(
            instruction_str=self.instruction_str,
            colunas_detalhes=self.descreve_colunas(),
            df_str=self.df.head(5).to_string(),
            query_str=query_str
        )
        resposta_llm = await self.llm.acomplete(prompt_formatado)
        return PandasCodeEvent(query_str=query_str, pandas_instructions=str(resposta_llm))

    # PASSO 2: Recebe o código (PandasCodeEvent), executa no computador e devolve o resultado (PandasOutputEvent)
    @step
    async def executar_codigo_pandas(self, ev: PandasCodeEvent) -> PandasOutputEvent:
        plt.close('all') 
        
        figura = None
        try:
            codigo_limpo = ev.pandas_instructions.replace("assistant:", "").replace("`", "").strip()
            pandas_output = eval(codigo_limpo, {"df": self.df})
        
            if len(plt.get_fignums()) > 0:
                figura = plt.gcf()
                
        except Exception as e:
            pandas_output = f"Erro ao executar o código Pandas: {e}"
            codigo_limpo = ev.pandas_instructions
            
        return PandasOutputEvent(
            query_str=ev.query_str,
            pandas_instructions=codigo_limpo,
            pandas_output=str(pandas_output),
            figura=figura 
        )

    # PASSO 3: Recebe o resultado (PandasOutputEvent) e devolve a resposta final para o usuário (StopEvent)
    @step
    async def sintetizar_resposta(self, ev: PandasOutputEvent) -> StopEvent:
        prompt_formatado = self.synthesis_prompt.format(
            query_str=ev.query_str,
            pandas_instructions=ev.pandas_instructions,
            pandas_output=ev.pandas_output
        )
        resposta_final = await self.llm.acomplete(prompt_formatado)

        return StopEvent(result={
            "texto": str(resposta_final),
            "grafico": ev.figura
        })

### Interface com o Gradio

In [ ]:
def load_data(file_path, df_state):
    if file_path is None or file_path == "":
        return "Faça o upload de um dataset para começar a análise", df_state
    try:
        df = pd.read_csv(file_path)
        return "Arquivo carregado com sucesso!", df
    except Exception as e:
        return f"Erro ao carregar o dataset: {str(e)}", df_state
    
async def proccess_question(question, df_state):
    if df_state is not None and question:
        fluxo = PandasAnalistaWorkflow(df=df_state, verbose=True)
        resposta = await fluxo.run(query=question)
        
        # Retorna o texto para a Textbox e o gráfico para o componente Plot
        return resposta["texto"], resposta["grafico"]
    
    return "Por favor, carregue um arquivo CSV primeiro!", None

with gr.Blocks() as app:
    in_file = gr.File(file_count="single", type="filepath", label="Upload CSV")
    status_upload = gr.Textbox(label="Status do Upload")
    in_question = gr.Textbox(label="Digite sua pergunta sobre os dados")
    bt_submit = gr.Button("Enviar")
    
    out_answer = gr.Textbox(label="Resposta")
    out_plot = gr.Plot(label="Visualização Gráfica") 
    
    df_state = gr.State(value=None)
    
    in_file.change(fn=load_data, inputs=[in_file, df_state], outputs=[status_upload, df_state])
    
    bt_submit.click(fn=proccess_question, 
                    inputs=[in_question, df_state], 
                    outputs=[out_answer, out_plot]) 

app.launch(debug=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


/Users/elianeorlandin/Documents/Desenvolvimento/IA-Project-Agent/tutorials/llamaindex2/.venv/lib/python3.13/site-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/elianeorlandin/Documents/Desenvolvimento/IA-Project-Agent/tutorials/llamaindex2/.venv/lib/python3.13/site-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


[tick] add: StartEvent(query='Exiba para mim um gráfico da distribuição das aval...')
[gerar_codigo_pandas:0] started from StartEvent
[gerar_codigo_pandas:0] complete with PandasCodeEvent
[tick] add: PandasCodeEvent(query_str='Exiba para mim um gráfico da distribuição das aval...', pandas_instructions="df['avaliacao'].plot.hist(bins=10, figsize=(10, 6)...")
[executar_codigo_pandas:0] started from PandasCodeEvent
[executar_codigo_pandas:0] complete with PandasOutputEvent
[tick] add: PandasOutputEvent(query_str='Exiba para mim um gráfico da distribuição das aval...', pandas_instructions="df['avaliacao'].plot.hist(bins=10, figsize=(10, 6)...", pandas_output='Axes(0.125,0.11;0.77...
[sintetizar_resposta:0] started from PandasOutputEvent
[result] StopEvent(result={'texto': 'O gráfico da distribuição das avaliações foi gerado com sucesso, mostrando a frequência de cada faixa de avaliação. O histograma apresenta 10 faixas de avaliação, com ta...
[sintetizar_resposta:0] complete with StopEvent

/Users/elianeorlandin/Documents/Desenvolvimento/IA-Project-Agent/tutorials/llamaindex2/.venv/lib/python3.13/site-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


[tick] add: StartEvent(query='Crie um gráfico de barras mostrando o faturamento ...')
[gerar_codigo_pandas:0] started from StartEvent
[gerar_codigo_pandas:0] complete with PandasCodeEvent
[tick] add: PandasCodeEvent(query_str='Crie um gráfico de barras mostrando o faturamento ...', pandas_instructions="df.groupby('cidade')['total'].sum().plot(kind='bar...")
[executar_codigo_pandas:0] started from PandasCodeEvent
[executar_codigo_pandas:0] complete with PandasOutputEvent
[tick] add: PandasOutputEvent(query_str='Crie um gráfico de barras mostrando o faturamento ...', pandas_instructions="df.groupby('cidade')['total'].sum().plot(kind='bar...", pandas_output='Axes(0.125,0.11;0.77...
[sintetizar_resposta:0] started from PandasOutputEvent
[result] StopEvent(result={'texto': "Foi gerado um gráfico de barras que exibe o faturamento total por cidade, permitindo uma visualização clara e direta dos dados. Esse tipo de gráfico é especialmente útil...
[sintetizar_resposta:0] complete with StopEvent

...

[tick] add: StartEvent(query='Mostre um gráfico de proporção (pizza) ilustrando ...')
[gerar_codigo_pandas:0] started from StartEvent
[gerar_codigo_pandas:0] complete with PandasCodeEvent
[tick] add: PandasCodeEvent(query_str='Mostre um gráfico de proporção (pizza) ilustrando ...', pandas_instructions="df['filial'].value_counts().plot(kind='pie', autop...")
[executar_codigo_pandas:0] started from PandasCodeEvent
[executar_codigo_pandas:0] complete with PandasOutputEvent
[tick] add: PandasOutputEvent(query_str='Mostre um gráfico de proporção (pizza) ilustrando ...', pandas_instructions="df['filial'].value_counts().plot(kind='pie', autop...", pandas_output='Axes(0.22375,0.11;0....
[sintetizar_resposta:0] started from PandasOutputEvent
[result] StopEvent(result={'texto': "Foi gerado um gráfico de pizza que ilustra a quantidade de itens vendidos por filial, demonstrando a proporção de vendas de cada uma. O gráfico fornece uma visualização ...
[sintetizar_resposta:0] complete with StopEvent

/Users/elianeorlandin/Documents/Desenvolvimento/IA-Project-Agent/tutorials/llamaindex2/.venv/lib/python3.13/site-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


[tick] add: StartEvent(query='Exiba para mim um gráfico da distribuição das aval...')
[gerar_codigo_pandas:0] started from StartEvent
[gerar_codigo_pandas:0] complete with PandasCodeEvent
[tick] add: PandasCodeEvent(query_str='Exiba para mim um gráfico da distribuição das aval...', pandas_instructions="df['avaliacao'].plot.hist(bins=10, figsize=(10, 6)...")
[executar_codigo_pandas:0] started from PandasCodeEvent
[executar_codigo_pandas:0] complete with PandasOutputEvent
[tick] add: PandasOutputEvent(query_str='Exiba para mim um gráfico da distribuição das aval...', pandas_instructions="df['avaliacao'].plot.hist(bins=10, figsize=(10, 6)...", pandas_output='Axes(0.125,0.11;0.77...
[sintetizar_resposta:0] started from PandasOutputEvent
[result] StopEvent(result={'texto': "Foi exibido um gráfico de histograma que representa a distribuição das avaliações. Esse gráfico é útil para visualizar a frequência com que cada avaliação ocorre, permit...
[sintetizar_resposta:0] complete with StopEvent

/Users/elianeorlandin/Documents/Desenvolvimento/IA-Project-Agent/tutorials/llamaindex2/.venv/lib/python3.13/site-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


[tick] add: StartEvent(query='Exiba para mim um gráfico da distribuição  da aval...')
[gerar_codigo_pandas:0] started from StartEvent
[gerar_codigo_pandas:0] complete with PandasCodeEvent
[tick] add: PandasCodeEvent(query_str='Exiba para mim um gráfico da distribuição  da aval...', pandas_instructions="df.groupby('filial')['avaliacao'].mean().plot(kind...")
[executar_codigo_pandas:0] started from PandasCodeEvent
[executar_codigo_pandas:0] complete with PandasOutputEvent
[tick] add: PandasOutputEvent(query_str='Exiba para mim um gráfico da distribuição  da aval...', pandas_instructions="df.groupby('filial')['avaliacao'].mean().plot(kind...", pandas_output='Axes(0.125,0.11;0.77...
[sintetizar_resposta:0] started from PandasOutputEvent
[result] StopEvent(result={'texto': "O gráfico da distribuição da avaliação média de cada filial foi gerado com sucesso. Nele, é possível visualizar a média das avaliações para cada uma das filiais, o que p...
[sintetizar_resposta:0] complete with StopEvent